# PDE Discovery: Nonlinear Diffusion (PME m=2)

f_t = f^2*Delta f + 2*f*|grad f|^2

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sys; sys.path.insert(0,'.')
from multigraph_operators import (
    build_equivariant_library, build_standard_library,
    TrainSTRidge, print_pde, rotation_90,
    spectral_laplacian, spectral_grad)
np.random.seed(42); print('imports ok')

imports ok


In [2]:
N=64; L=6*np.pi
x=np.linspace(0,L,N,endpoint=False)
X,Y=np.meshgrid(x,x,indexing='ij')
dt=0.002; T_end=2.; n_steps=int(T_end/dt); save_every=25
cx,cy=L/2,L/2; sigma=L/8
f0=np.exp(-((X-cx)**2+(Y-cy)**2)/(2*sigma**2))
f0=f0/f0.max()
print('setup ok')

setup ok


In [3]:
snapshots=[]; times=[]
f=f0.copy()
for step in range(n_steps):
    Df=spectral_laplacian(f,L,L)
    fx,fy=spectral_grad(f,L,L)
    rhs=f**2*Df+2*f*(fx**2+fy**2)
    f=np.clip(f+dt*rhs,0,None)
    if step%save_every==0:
        snapshots.append(f.copy()); times.append(step*dt)
snapshots=np.array(snapshots); times=np.array(times)
print(f'Snapshots:{len(snapshots)}, f in [{snapshots.min():.3f},{snapshots.max():.3f}]')

Snapshots:40, f in [0.000,0.999]


In [4]:
fig,axes=plt.subplots(1,4,figsize=(16,4))
vmax=snapshots[0].max()
for ax,idx in zip(axes,[0,len(snapshots)//4,len(snapshots)//2,-1]):
    im=ax.imshow(snapshots[idx],cmap='hot_r',vmin=0,vmax=vmax,origin='lower',extent=[0,L,0,L])
    ax.set_title(f't={times[idx]:.2f}',fontsize=13); ax.axis('off')
fig.colorbar(im,ax=axes[-1],shrink=0.8)
fig.suptitle('Nonlinear diffusion PME (m=2)',fontsize=14)
plt.tight_layout(); plt.savefig('../figures/nonlinear_diffusion_dynamics.pdf',bbox_inches='tight'); plt.close()
print('saved dynamics')

saved dynamics


In [5]:
threshold=0.05
dt_snap=times[1]-times[0]
ft_snaps=(snapshots[2:]-snapshots[:-2])/(2*dt_snap)
f_snaps=snapshots[1:-1]; print(f'{len(f_snaps)} snapshots for regression')

38 snapshots for regression


In [6]:
n_per_snap=200; Teq_rows=[]; Tstd_rows=[]; Ft_rows=[]
np.random.seed(0)
for s in range(len(f_snaps)):
    mask=(f_snaps[s]>threshold).ravel(); avail=np.where(mask)[0]
    ns=min(n_per_snap,len(avail))
    idx=np.random.choice(avail,ns,replace=False)
    Ft_rows.append(ft_snaps[s].ravel()[idx])
    Te,desc_eq=build_equivariant_library(f_snaps[s],L,L)
    Ts,desc_std=build_standard_library(f_snaps[s],L,L,max_order=4,max_poly=3)
    Teq_rows.append(Te[idx]); Tstd_rows.append(Ts[idx])
Ft_all=np.concatenate(Ft_rows)
Theta_eq=np.vstack(Teq_rows).astype(float)
Theta_std=np.vstack(Tstd_rows).astype(float)
print(f'Theta_eq:{Theta_eq.shape},cond={np.linalg.cond(Theta_eq):.2e}')
print(f'Theta_std:{Theta_std.shape},cond={np.linalg.cond(Theta_std):.2e}')

Theta_eq:(7600, 20),cond=1.14e+05
Theta_std:(7600, 60),cond=2.23e+04


In [7]:
lam=1e-5; d_tol=0.05
print('=== Equivariant ==='); w_eq=TrainSTRidge(Theta_eq,Ft_all,lam,d_tol); print_pde(w_eq,desc_eq)
print('\n=== Standard ==='); w_std=TrainSTRidge(Theta_std,Ft_all,lam,d_tol); print_pde(w_std,desc_std)

=== Equivariant ===
f_t = (+1.00108)f²·Δf + (+2.00313)f·|∇f|²

=== Standard ===
f_t = (-0.05049)1 + (+0.21434)f_yyyy + (+0.22653)f_xxxx + (+0.47515)f + (-2.12683)f·f_yyyy + (-2.23383)f·f_xxxx + (+0.13465)f^2 + (+4.07529)f^2·f_yy + (+4.18320)f^2·f_xx + (+6.15804)f^2·f_yyyy + (+6.38603)f^2·f_xxxx + (-0.80121)f^3 + (-3.70865)f^3·f_yy + (-4.10333)f^3·f_xx + (-3.88642)f^3·f_yyyy + (-3.89828)f^3·f_xxxx


In [8]:
true_eq=np.zeros(len(desc_eq)); true_eq[6]=1.; true_eq[8]=2.
act=np.abs(true_eq)>1e-10
coeff_err=float(np.max(np.abs(w_eq[act]-true_eq[act])))
fp=int(np.sum((np.abs(w_eq)>1e-10)&~act))
fn=int(np.sum((np.abs(w_eq)<1e-10)&act))
print(f'True: f2Df->1.0, f|grad|2->2.0')
print(f'coeff_error={coeff_err:.3e}, false_pos={fp}, false_neg={fn}')

True: f2Df->1.0, f|grad|2->2.0
coeff_error=3.131e-03, false_pos=0, false_neg=0


In [9]:
nl=0.02
snaps_n=np.clip(snapshots+nl*snapshots.std()*np.random.randn(*snapshots.shape),0,None)
ft_n=(snaps_n[2:]-snaps_n[:-2])/(2*dt_snap); f_n=snaps_n[1:-1]
np.random.seed(0); Ftn=[]; Teqn=[]; Tstdn=[]
for s in range(len(f_n)):
    mask=(f_n[s]>threshold).ravel(); avail=np.where(mask)[0]; ns=min(n_per_snap,len(avail))
    idx=np.random.choice(avail,ns,replace=False); Ftn.append(ft_n[s].ravel()[idx])
    Te,_=build_equivariant_library(f_n[s],L,L); Ts,_=build_standard_library(f_n[s],L,L,max_order=4,max_poly=3)
    Teqn.append(Te[idx]); Tstdn.append(Ts[idx])
Ftn=np.concatenate(Ftn); Teqn=np.vstack(Teqn).astype(float); Tstdn=np.vstack(Tstdn).astype(float)
w_eq_n=TrainSTRidge(Teqn,Ftn,lam,d_tol); w_std_n=TrainSTRidge(Tstdn,Ftn,lam,d_tol)
print('Equivariant noisy:'); print_pde(w_eq_n,desc_eq)
print('Standard noisy:'); print_pde(w_std_n,desc_std)

Equivariant noisy:
f_t = (+0.00952)1 + (-0.24128)f + (+0.76340)f^2 + (-0.79058)f^3 + (+0.02238)Δf + (-0.11909)f·Δf + (+0.10377)f²·Δf + (+1.05884)|∇f|² + (-1.10179)f²·|∇f|² + (+0.00018)Δ²f + (-0.00088)f·Δ²f + (+0.00079)f²·Δ²f + (+0.01207)⟨∇Δf,∇f⟩
Standard noisy:
f_t = (+0.64545)f + (-1.39548)f^2 + (+0.49781)f^3


In [10]:
ftr=(rotation_90(snapshots[2:].transpose(1,2,0)).transpose(2,0,1)
     -rotation_90(snapshots[:-2].transpose(1,2,0)).transpose(2,0,1))/(2*dt_snap)
fr=rotation_90(snapshots[1:-1].transpose(1,2,0)).transpose(2,0,1)
np.random.seed(0); Ftr=[]; Teqr=[]; Tstdr=[]
for s in range(len(fr)):
    mask=(fr[s]>threshold).ravel(); avail=np.where(mask)[0]; ns=min(n_per_snap,len(avail))
    idx=np.random.choice(avail,ns,replace=False); Ftr.append(ftr[s].ravel()[idx])
    Te,_=build_equivariant_library(fr[s],L,L); Ts,_=build_standard_library(fr[s],L,L,max_order=4,max_poly=3)
    Teqr.append(Te[idx]); Tstdr.append(Ts[idx])
Ftr=np.concatenate(Ftr); Teqr=np.vstack(Teqr).astype(float); Tstdr=np.vstack(Tstdr).astype(float)
w_eq_r=TrainSTRidge(Teqr,Ftr,lam,d_tol); w_std_r=TrainSTRidge(Tstdr,Ftr,lam,d_tol)
print(f'Equivariant rot change: {np.max(np.abs(w_eq-w_eq_r)):.2e}')
print(f'Standard rot change:    {np.max(np.abs(w_std-w_std_r)):.2e}')

Equivariant rot change: 3.55e-15
Standard rot change:    7.72e-12


In [11]:
bc=['tab:blue' if i in [6,8] and abs(c)>1e-10 else 'tab:red' if abs(c)>1e-10 else 'tab:gray'
    for i,c in enumerate(w_eq)]
fig,axes=plt.subplots(1,2,figsize=(16,4))
ax=axes[0]; ax.bar(range(len(w_eq)),np.real(w_eq),color=bc)
ax.set_xticks(range(len(desc_eq))); ax.set_xticklabels(desc_eq,rotation=45,ha='right',fontsize=9)
ax.set_title('Equivariant coefficients'); ax.set_ylabel('value'); ax.axhline(0,color='k',lw=0.5)
ax=axes[1]; nz=np.abs(w_std)>1e-10
if nz.any():
    ax.bar(np.where(nz)[0],np.real(w_std[nz]),color='tab:red')
    ax.set_xticks(np.where(nz)[0]); ax.set_xticklabels([desc_std[i] for i in np.where(nz)[0]],rotation=45,ha='right',fontsize=9)
ax.set_title('Standard coefficients (active only)'); ax.set_ylabel('value'); ax.axhline(0,color='k',lw=0.5)
plt.tight_layout(); plt.savefig('../figures/nonlinear_diffusion_coefficients.pdf',bbox_inches='tight'); plt.close()
print('saved coefficients')

saved coefficients


In [12]:
np.save('nonlinear_diffusion_results.npy',{
    'w_eq':w_eq,'w_std':w_std,'w_eq_noisy':w_eq_n,'w_std_noisy':w_std_n,
    'w_eq_rotated':w_eq_r,'w_std_rotated':w_std_r,
    'cond_eq':float(np.linalg.cond(Theta_eq)),'cond_std':float(np.linalg.cond(Theta_std)),
    'true_eq':true_eq,'desc_eq':desc_eq,'desc_std':desc_std,
    'coeff_error':coeff_err,'false_pos':fp,
    'rot_change_eq':float(np.max(np.abs(w_eq-w_eq_r))),
    'rot_change_std':float(np.max(np.abs(w_std-w_std_r))),
},allow_pickle=True); print('Saved nonlinear_diffusion_results.npy')

Saved nonlinear_diffusion_results.npy
